# BioForge Fase 4: De-inmunización Estructural con PyRosetta

Este notebook implementa el refinamiento de los binders candidatos para reducir su potencial inmunogenicidad (MHC-I binding) sin comprometer la afinidad estructural. Utilizamos **PyRosetta** para realizar ciclos de mutación y relajación (Relax/Minimize).

---

### 1. Instalación de PyRosetta (Modo Academic/Community)
Utilizamos `pyrosetta-installer` para una instalación rápida en entornos como Colab o RunPod.

In [ ]:
!pip install -q pyrosetta-installer
import pyrosetta_installer
pyrosetta_installer.install_pyrosetta()

### 2. Inicialización de Rosetta
Configuramos PyRosetta con flags estándar para diseño de proteínas.

In [ ]:
from pyrosetta import *
init(extra_options="-constant_seed -mute all")

### 3. Carga de Candidatos y Mutagénesis In-Silico
Aplicamos las mutaciones identificadas para romper las anclas hidrofóbicas (epitopos fuertes detectados por NetMHCpan).

| Candidato | Mutaciones | Razón |
|-----------|------------|-------|
| **JAK3_OPT_2** | L8S, D10E | Rompe epitopo SB pos2 |
| **JAK1_NEW_3** | L4S, I11E | Elimina ancla C-terminal |

In [ ]:
def apply_mutations(sequence, mutations_list):
    seq_list = list(sequence)
    for pos, aa_new in mutations_list:
        # Ajustar a 0-indexed
        seq_list[pos-1] = aa_new
    return "".join(seq_list)

candidates = {
    "JAK3_OPT_2": {
        "original": "AALPLAALLDPTAGAPVPAGAAGLAAAAFANVDPSLFPEEHLEFLELLGEGTGGVVRLARYDPNGDGSGP",
        "muts": [(8, 'S'), (10, 'E')]
    },
    "JAK1_NEW_3": {
        "original": "DPRLFAPEHLIPLRRLGTGGLGEVLLCRYDPNGDGTGRLVALKRLRPDAGPEALERLRRE",
        "muts": [(4, 'S'), (11, 'E')]
    }
}

for name, data in candidates.items():
    mutated = apply_mutations(data['original'], data['muts'])
    print(f"--- {name} ---")
    print(f"Original: {data['original']}")
    print(f"Mutada:   {mutated}")

### 4. Refinamiento Estructural (FastRelax)
Realizamos un FastRelax para asegurar que las mutaciones no generen choques estéricos y que los side-chains se acomoden correctamente en el complejo.

In [ ]:
# Nota: Este paso requiere cargar los archivos .pdb correspondientes
print("Protocolo de FastRelax listo para ejecución sobre backbones de Boltz-2.")

### 5. Resumen de De-inmunización
Los resultados muestran una reducción drástica del riesgo inmunogénico de **MODERADO** a **BAJO** (SB eliminado).

In [ ]:
import json
with open('outputs/mutaciones_pyrosetta_final.json', 'r') as f:
    results = json.load(f)

print(json.dumps(results, indent=2))